# 4.29 人民大学各专业填报热度分析

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Step 1. 获得所有省份2023年的CSV文件列表
root_path = Path('../pybook-data/ruc_data')
csv_files = list(root_path.rglob('2023.csv'))
csv_files = [str(x) for x in csv_files]

In [3]:
# Step 2. 读入各省的招生数据表并清洗数据
tables = [None]*len(csv_files)
for i,csv_fname in enumerate(csv_files):
    prov = csv_fname.split('/')[-2]
    df = pd.read_csv(csv_fname).rename(columns={'理工/物理类 最低':'最低分'})
    sub_table = df[['专业名称','最低分']].drop_duplicates() #去掉可能存在的重复记录 
    sub_table.insert(0, '省份', [prov]*len(sub_table))
    tables[i] = sub_table.dropna() # 去掉没有理工类的记录

In [4]:
# Step 3. 将各省各专业投档分纵向拼接成一个长表
long_table = pd.concat(tables, ignore_index=True)
print('记录数:', len(long_table)) # 记录数: 537

记录数: 537


In [5]:
# Step 4. 透视重组，获得一个宽表，每行代表一个省份，每列对应一个专业
wide_table = long_table.pivot(index='省份', columns='专业名称', values='最低分')

In [6]:
# Step 5. 专业热度统计
wide_table.fillna(0, inplace=True) #先用0填充缺失值
wide_table['最高分专业'] = wide_table.idxmax(axis=1) #找出每个省（直辖市）投档线最高的专业
wide_table['最高分'] = wide_table.iloc[:, :-1].max(axis=1) #单独列出最热门专业的投档分
wide_table['最高分专业'].value_counts().to_frame() # 统计热门专业的省份频率

,count
最高分专业,
法学,10
金融学类,7
人工智能（拔尖班）,6
数学类（金融学与数学双主学位项目）,2
统计学类,1
政治学、经济学与哲学（PPE实验班）,1
经济学类（国家专项）,1
经济学类,1
社会科学试验班（管理学科类，国家专项，区外）,1


# 4.30 按地区频度校正之后的专业热度估计

In [7]:
s1 = wide_table['最高分专业'].value_counts()
s2 = (wide_table!=0).sum()
s1.name = '热度'
s2.name = '地区频度'
overview = pd.concat([s1, s2], axis=1).dropna()
overview['占比'] = overview[s1.name] / overview[s2.name]
overview.sort_values(by='占比', ascending=False)

,热度,地区频度,占比
社会科学试验班（管理学科类，国家专项，区外）,1.0,1,1.000000
经济学类（国家专项）,1.0,2,0.500000
法学,10.0,24,0.416667
数学类（金融学与数学双主学位项目）,2.0,6,0.333333
人工智能（拔尖班）,6.0,23,0.260870
金融学类,7.0,29,0.241379
政治学、经济学与哲学（PPE实验班）,1.0,6,0.166667
经济学类,1.0,27,0.037037
统计学类,1.0,28,0.035714


# 4.31 按占比对专业进行排序

In [8]:
bar = int(len(wide_table)*0.3) # 只考虑在全国30%以上省份投放的专业
overview = overview[ overview['地区频度']>=bar ]
overview.sort_values(by='占比', ascending=False)

,热度,地区频度,占比
法学,10.0,24,0.416667
人工智能（拔尖班）,6.0,23,0.260870
金融学类,7.0,29,0.241379
经济学类,1.0,27,0.037037
统计学类,1.0,28,0.035714
